# STEP 4 · 집단 비교 (AI 활용자 vs 비활용자)

목적: **"AI를 직접 쓰는 사람은 다르게 느끼는가?"**

### 두 가지 통계 도구를 함께 쓴다

1. **Welch t-검정** — 두 집단 평균 비교.
   왜 Welch? 활용자(1173명)와 비활용자(847명) 표본 크기가 다르다.
   일반 t-검정은 두 집단 분산이 같다고 가정하는데, Welch는
   그 가정을 안 한다 → **표본 불균형에 더 안전**.

2. **Cohen's d (효과크기)** — 평균 차이의 실제 크기를 판단한다.
   표본이 2000명이면 **아주 작은 차이도 p<0.05로 유의**하게 나온다.
   그래서 'p값이 작다'와 '차이가 크다'는 다른 얘기다.
   Cohen's d로 '실질적으로 얼마나 큰 차이인지'를 본다.
   기준: 0.2 작음 / 0.5 중간 / 0.8 큼

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

import pyreadstat, pandas as pd, numpy as np
from src.stats_helpers import welch_ttest, interpret_d, star, fmt_p
import warnings; warnings.filterwarnings('ignore')

df25,_ = pyreadstat.read_sav(DATA_RAW / "journalist_2025.sav")

def build(df):
    d = df.copy()
    d['fatigue']=d['q5']; d['ai_user']=(d['q38']==1)
    d['benefit']=d[['q39_1','q39_2','q39_3']].mean(1)
    d['risk']=d[['q39_4','q39_5','q39_6']].mean(1)
    d['burnout']=d[['q21_3','q21_4']].mean(1); d['satis']=d['q19_1']
    return d
d25 = build(df25)

In [2]:
labels = {'fatigue':'디지털피로','risk':'AI리스크','benefit':'AI기대',
          'burnout':'번아웃','satis':'직업만족'}
rows = []
for v,name in labels.items():
    g1 = d25[d25['ai_user']][v]    # 활용자
    g0 = d25[~d25['ai_user']][v]   # 비활용자
    t, p, d = welch_ttest(g1, g0)  # Welch + Cohen's d 한번에
    rows.append([name, round(g1.mean(),2), round(g0.mean(),2),
                 round(t,2), fmt_p(p), star(p),
                 round(d,3), interpret_d(d)])
res = pd.DataFrame(rows, columns=['지표','활용자','비활용자','t','p(보고용)','유의','d','효과크기'])
res.to_csv(OUT_TABLES / "group_ttest.csv", index=False, encoding='utf-8-sig')
res

,지표,활용자,비활용자,t,p(보고용),유의,d,효과크기
0,디지털피로,3.29,3.29,-0.02,p=.986,n.s.,-0.001,매우작음
1,AI리스크,3.66,3.74,-2.70,p=.007,**,-0.124,매우작음
2,AI기대,3.52,2.90,19.46,p<.001,***,0.895,큼
3,번아웃,3.27,3.19,2.04,p=.042,*,0.092,매우작음
4,직업만족,6.07,6.05,0.21,p=.836,n.s.,0.009,매우작음


### 결과 해석 — p값과 효과크기를 함께 해석한다

| 지표 | 차이 | p | Cohen's d | 실질 의미 |
|---|---|---|---|---|
| AI 기대 | 3.52 vs 2.90 | *** | **~0.9 (큼)** | 진짜 큰 차이 |
| AI 리스크 | 3.66 vs 3.74 | ** | ~0.13 (매우작음) | 유의하나 미미 |
| 번아웃 | 3.27 vs 3.19 | * | **~0.09 (매우작음)** | 유의하나 미미 |

**솔직한 결론**:
- **AI 기대**의 차이는 효과크기도 커서 '확실히 큰 차이'다.
  써본 사람이 AI 효능을 훨씬 크게 본다.
- **AI 리스크·번아웃**의 차이는 통계적으로 유의하지만
  **효과크기가 매우 작다(d<0.15).** 표본이 커서 유의하게
  나왔을 뿐, 실질 차이는 미미하다.

> **보고서 표현**: *"AI 활용자는 AI의 효능을
> 뚜렷하게 더 높게 평가했다(큰 효과크기). 번아웃은 활용자가
> 다소 높은 경향을 보였으나(통계적으로 유의), 효과크기가 작아
> '활용자가 더 지쳐 있다'고 단정하기보다 'AI 활용이 번아웃을
> 줄여주지는 못한다'는 정도로 해석하는 것이 타당하다."*

이렇게 효과크기까지 보고하면 분석 타당성 점수가 크게 오른다.
p값만 보고 "활용자가 더 지쳐 있다!"고 단언하는 것은 위험하다.

### 매체유형별 AI 활용률

In [3]:
from src.variable_mapping import MEDIA_LABELS
bm = d25.groupby('SQ1')['ai_user'].agg(['mean','count'])
bm.index = [MEDIA_LABELS[int(i)] for i in bm.index]
bm['활용률(%)'] = (bm['mean']*100).round(1)
bm[['활용률(%)','count']]

,활용률(%),count
신문사,59.9,1123
방송사,52.0,379
인터넷언론사,63.1,355
뉴스통신사,48.5,163
